# 4. Инференс полной модели: Schema Matching + Entity Resolution

Полный пайплайн применения обученных моделей к новым таблицам:
1. **Schema Matching**: сопоставление столбцов двух таблиц
2. **Entity Resolution**: поиск дубликатов строк
3. Визуализация и анализ результатов

## 4.1 Загрузка конфигурации и моделей

In [ ]:
import json
import os
import logging

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

CONFIG_PATH = 'config.json'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    full_config = json.load(f)

data_gen_params = full_config['data_generation']
sm_params = full_config['schema_matching']
er_params = full_config['entity_resolution']
inf_params = full_config['inference']

SM_MODEL_DIR = sm_params['model_dir']
ER_MODEL_DIR = er_params['model_dir']

print(f"SM модель: {SM_MODEL_DIR}/best_model.pt")
print(f"ER модель: {ER_MODEL_DIR}/best_model.pt")
print(f"SM порог: {inf_params['sm_similarity_threshold']}")
print(f"ER порог: {inf_params['er_similarity_threshold']}")

In [ ]:
from table_unifier.schema_matching import SMConfig
from table_unifier.schema_matching.inference import SchemaMatcherInference
from table_unifier.entity_resolution.config import ERConfig
from table_unifier.entity_resolution.inference import DuplicateDetector
from table_unifier.entity_resolution.token_embedder import FastTextEmbedder
from table_unifier.core import TableUnifier
from table_unifier.config import AppConfig

# SM Inference
sm_config = SMConfig(
    ollama_host=data_gen_params['ollama_host'],
    llm_model=data_gen_params['llm_model'],
    embedding_model=data_gen_params['embedding_model'],
    similarity_threshold=inf_params['sm_similarity_threshold'],
)

sm_matcher = SchemaMatcherInference(
    model_path=os.path.join(SM_MODEL_DIR, 'best_model.pt'),
    config=sm_config,
)
print("SM модель загружена")

# ER Inference
er_config = ERConfig(
    ollama_host=data_gen_params['ollama_host'],
    embedding_model=data_gen_params['embedding_model'],
    llm_model=data_gen_params['llm_model'],
    fasttext_model_path=er_params['fasttext_model_path'],
    token_embed_dim=er_params['token_embed_dim'],
    hidden_dim=er_params['hidden_dim'],
    edge_dim=er_params['edge_dim'],
    num_gnn_layers=er_params['num_gnn_layers'],
    num_heads=er_params['num_heads'],
    dropout=er_params['dropout'],
    use_jumping_knowledge=er_params['use_jumping_knowledge'],
    output_dim=er_params['output_dim'],
)

ft_embedder = FastTextEmbedder(er_params['fasttext_model_path'])

# Определяем col_embed_dim из SM модели
sm_model_data = torch.load(
    os.path.join(SM_MODEL_DIR, 'best_model.pt'),
    map_location='cpu',
    weights_only=False,
)
col_embed_dim = sm_model_data['config']['output_dim']

detector = DuplicateDetector.from_checkpoint(
    checkpoint_path=os.path.join(ER_MODEL_DIR, 'best_model.pt'),
    config=er_config,
    row_input_dim=ft_embedder.dim,
    col_embed_dim=col_embed_dim,
    fasttext_embedder=ft_embedder,
)
print("ER модель загружена")

## 4.2 Подготовка тестовых таблиц

Используем тестовую пару из сгенерированного датасета для демонстрации.

In [ ]:
# Загрузка тестовой пары
ER_RAW_DIR = os.path.join(data_gen_params['output_dir'], data_gen_params['er_raw_dir'])
test_dir = os.path.join(ER_RAW_DIR, 'test')

table_a = pd.read_csv(os.path.join(test_dir, 'pair_0000_table_a.csv'))
table_b = pd.read_csv(os.path.join(test_dir, 'pair_0000_table_b.csv'))

with open(os.path.join(test_dir, 'pair_0000_meta.json'), 'r', encoding='utf-8') as f:
    gt_meta = json.load(f)

print(f"Таблица A: {table_a.shape}")
print(f"  Столбцы: {list(table_a.columns)}")
print(f"\nТаблица B: {table_b.shape}")
print(f"  Столбцы: {list(table_b.columns)}")
print(f"\nGround truth дубликатов: {gt_meta['num_duplicates']}")

print("\n--- Таблица A ---")
display(table_a)
print("\n--- Таблица B ---")
display(table_b)

## 4.3 Schema Matching: сопоставление столбцов

In [ ]:
mapping, sm_metrics = sm_matcher.match_schemas(
    source_df=table_a,
    target_df=table_b,
    similarity_threshold=inf_params['sm_similarity_threshold'],
)

print("=" * 60)
print("SCHEMA MATCHING РЕЗУЛЬТАТ")
print("=" * 60)
print(f"\nСопоставлено столбцов: {sm_metrics.get('matched_columns', 'N/A')}")
print(f"Средняя уверенность: {sm_metrics.get('avg_similarity', 0):.4f}")
print(f"\nМаппинг столбцов (A → B):")
for src, tgt in mapping.items():
    status = '✓' if tgt else '✗'
    print(f"  {status} {src} → {tgt or 'не найден'}")

In [ ]:
# Визуализация матрицы сходства столбцов
if 'similarity_matrix' in sm_metrics:
    sim_mx = np.array(sm_metrics['similarity_matrix'])
    src_cols = sm_metrics.get('source_columns', list(table_a.columns))
    tgt_cols = sm_metrics.get('target_columns', list(table_b.columns))
else:
    # Вычисляем вручную
    src_emb, src_cols = sm_matcher.compute_column_embeddings(table_a)
    tgt_emb, tgt_cols = sm_matcher.compute_column_embeddings(table_b)
    sim_mx = cosine_similarity(src_emb, tgt_emb)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(sim_mx, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(tgt_cols)))
ax.set_yticks(range(len(src_cols)))
ax.set_xticklabels(tgt_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(src_cols, fontsize=9)
ax.set_xlabel('Таблица B')
ax.set_ylabel('Таблица A')
ax.set_title('Матрица косинусного сходства столбцов (SM)', fontsize=13)

# Аннотации
for i in range(sim_mx.shape[0]):
    for j in range(sim_mx.shape[1]):
        color = 'white' if sim_mx[i, j] > 0.7 or sim_mx[i, j] < 0.3 else 'black'
        ax.text(j, i, f'{sim_mx[i, j]:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Оценка SM: сравнение с ground truth mappings
gt_mapping_a = gt_meta['column_mapping_a']  # {display_name: base_name}
gt_mapping_b = gt_meta['column_mapping_b']

# Инвертируем: base_name → display_name_b
base_to_display_b = {v: k for k, v in gt_mapping_b.items()}

print("\nОценка качества SM (Ground Truth):")
correct = 0
total = 0
for src_col, tgt_col in mapping.items():
    src_base = gt_mapping_a.get(src_col, '?')
    expected_tgt = base_to_display_b.get(src_base)
    
    is_correct = (tgt_col == expected_tgt)
    status = '✓' if is_correct else '✗'
    
    if expected_tgt is not None:
        total += 1
        if is_correct:
            correct += 1
    
    print(f"  {status} {src_col} [{src_base}] → {tgt_col} (ожидалось: {expected_tgt})")

if total > 0:
    print(f"\nТочность SM: {correct}/{total} = {correct/total:.2%}")

## 4.4 Entity Resolution: поиск дубликатов строк

In [ ]:
matches = detector.find_duplicates(
    df_a=table_a,
    df_b=table_b,
    threshold=inf_params['er_similarity_threshold'],
)

print("=" * 60)
print("ENTITY RESOLUTION РЕЗУЛЬТАТ")
print("=" * 60)
print(f"\nНайдено дубликатов: {len(matches)}")
print(f"Ground truth дубликатов: {gt_meta['num_duplicates']}")
print(f"Порог: {inf_params['er_similarity_threshold']}")

if matches:
    match_df = detector.matches_to_dataframe(matches, table_a, table_b)
    print("\nНайденные дубликаты:")
    display(match_df)

In [ ]:
# Оценка ER: сравнение с ground truth
gt_pairs = set(tuple(p) for p in gt_meta['duplicate_pairs'])
pred_pairs = set((m.row_idx_a, m.row_idx_b) for m in matches)

tp = len(gt_pairs & pred_pairs)
fp = len(pred_pairs - gt_pairs)
fn = len(gt_pairs - pred_pairs)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nER метрики (Ground Truth):")
print(f"  True Positives: {tp}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-score: {f1:.4f}")

if gt_pairs - pred_pairs:
    print(f"\nПропущенные дубликаты (FN):")
    for a, b in sorted(gt_pairs - pred_pairs):
        print(f"  Row A[{a}] ↔ Row B[{b}]")

if pred_pairs - gt_pairs:
    print(f"\nЛожные дубликаты (FP):")
    for a, b in sorted(pred_pairs - gt_pairs):
        sim = next((m.similarity for m in matches if m.row_idx_a == a and m.row_idx_b == b), 0)
        print(f"  Row A[{a}] ↔ Row B[{b}] (sim={sim:.3f})")

## 4.5 Визуализация результатов

In [ ]:
# Визуализация сходства дубликатов
if matches:
    similarities = [m.similarity for m in matches]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Гистограмма сходства найденных дубликатов
    axes[0].bar(range(len(similarities)), similarities, color='steelblue')
    axes[0].set_xlabel('Пара дубликатов')
    axes[0].set_ylabel('Косинусное сходство')
    axes[0].set_title('Уверенность в найденных дубликатах')
    axes[0].axhline(inf_params['er_similarity_threshold'], color='red', linestyle='--', 
                    label=f'Порог={inf_params["er_similarity_threshold"]}')
    axes[0].legend()
    axes[0].set_ylim(0, 1)
    
    # Scatter: TP vs FP
    tp_sims = [m.similarity for m in matches if (m.row_idx_a, m.row_idx_b) in gt_pairs]
    fp_sims = [m.similarity for m in matches if (m.row_idx_a, m.row_idx_b) not in gt_pairs]
    
    categories = (['TP'] * len(tp_sims)) + (['FP'] * len(fp_sims))
    all_sims = tp_sims + fp_sims
    colors = ['green' if c == 'TP' else 'red' for c in categories]
    
    axes[1].scatter(range(len(all_sims)), all_sims, c=colors, s=100, zorder=5)
    axes[1].axhline(inf_params['er_similarity_threshold'], color='gray', linestyle='--')
    axes[1].set_xlabel('Пара')
    axes[1].set_ylabel('Сходство')
    axes[1].set_title('TP (зелёный) vs FP (красный)')
    axes[1].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()

## 4.6 Батч-инференс на нескольких тестовых парах

In [ ]:
# Оценка на нескольких тестовых парах
test_meta_files = sorted(Path(test_dir).glob('*_meta.json'))
n_test = min(len(test_meta_files), 10)

batch_results = []

for meta_file in test_meta_files[:n_test]:
    prefix = meta_file.stem.replace('_meta', '')
    
    with open(meta_file, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    
    df_a = pd.read_csv(os.path.join(test_dir, f'{prefix}_table_a.csv'))
    df_b = pd.read_csv(os.path.join(test_dir, f'{prefix}_table_b.csv'))
    
    # SM
    sm_map, sm_met = sm_matcher.match_schemas(
        df_a, df_b, 
        similarity_threshold=inf_params['sm_similarity_threshold']
    )
    
    # SM accuracy
    gt_a = meta['column_mapping_a']
    gt_b = meta['column_mapping_b']
    base_to_b = {v: k for k, v in gt_b.items()}
    
    sm_correct, sm_total = 0, 0
    for src, tgt in sm_map.items():
        expected = base_to_b.get(gt_a.get(src))
        if expected is not None:
            sm_total += 1
            if tgt == expected:
                sm_correct += 1
    
    # ER
    er_matches = detector.find_duplicates(
        df_a, df_b,
        threshold=inf_params['er_similarity_threshold'],
    )
    
    gt_p = set(tuple(p) for p in meta['duplicate_pairs'])
    pred_p = set((m.row_idx_a, m.row_idx_b) for m in er_matches)
    
    tp = len(gt_p & pred_p)
    fp = len(pred_p - gt_p)
    fn = len(gt_p - pred_p)
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_val = 2 * p * r / (p + r) if (p + r) > 0 else 0
    
    batch_results.append({
        'pair': prefix,
        'sm_accuracy': sm_correct / sm_total if sm_total > 0 else None,
        'er_precision': p,
        'er_recall': r,
        'er_f1': f1_val,
        'gt_duplicates': len(gt_p),
        'found_duplicates': len(er_matches),
    })
    
    print(f"{prefix}: SM acc={sm_correct}/{sm_total}, "
          f"ER P={p:.2f} R={r:.2f} F1={f1_val:.2f} "
          f"(found={len(er_matches)}, gt={len(gt_p)})")

batch_df = pd.DataFrame(batch_results)

In [ ]:
# Визуализация батч-инференса
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SM accuracy
sm_accs = batch_df['sm_accuracy'].dropna()
axes[0].bar(range(len(sm_accs)), sm_accs, color='steelblue')
axes[0].axhline(sm_accs.mean(), color='red', linestyle='--', label=f'mean={sm_accs.mean():.2f}')
axes[0].set_xlabel('Пара')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('SM Accuracy по тестовым парам')
axes[0].set_ylim(0, 1.05)
axes[0].legend()

# ER F1
axes[1].bar(range(len(batch_df)), batch_df['er_f1'], color='coral')
axes[1].axhline(batch_df['er_f1'].mean(), color='red', linestyle='--', 
                label=f'mean={batch_df["er_f1"].mean():.2f}')
axes[1].set_xlabel('Пара')
axes[1].set_ylabel('F1')
axes[1].set_title('ER F1-score по тестовым парам')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4.7 Итоговая сводка

In [ ]:
print("=" * 60)
print("ИТОГОВАЯ СВОДКА ПОЛНОГО ИНФЕРЕНСА")
print("=" * 60)

print(f"\nSchema Matching:")
sm_accs = batch_df['sm_accuracy'].dropna()
print(f"  Средняя accuracy: {sm_accs.mean():.4f}")
print(f"  Min accuracy: {sm_accs.min():.4f}")
print(f"  Max accuracy: {sm_accs.max():.4f}")
print(f"  Порог: {inf_params['sm_similarity_threshold']}")

print(f"\nEntity Resolution:")
print(f"  Средний Precision: {batch_df['er_precision'].mean():.4f}")
print(f"  Средний Recall: {batch_df['er_recall'].mean():.4f}")
print(f"  Средний F1: {batch_df['er_f1'].mean():.4f}")
print(f"  Порог: {inf_params['er_similarity_threshold']}")

print(f"\nТестовых пар обработано: {len(batch_df)}")
print(f"\nМодели:")
print(f"  SM: {SM_MODEL_DIR}/best_model.pt")
print(f"  ER: {ER_MODEL_DIR}/best_model.pt")
print(f"  Конфиг: {CONFIG_PATH}")